<a href="https://colab.research.google.com/github/Thangamuniyandi894/mini_project_3/blob/main/ML_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
df=pd.read_csv("/content/Hr_job_analysis.csv")

In [ ]:
df.shape

# **ENCODING**

In [ ]:
encode=OrdinalEncoder()

for i in df.select_dtypes(include=["object"]).columns:
  df[i]=encode.fit_transform(df[[i]])

In [ ]:
df

**CORRELATION TEST**

In [ ]:
df.corr()

In [ ]:

corr_with_target = df.corr()['status'].abs()


threshold = 0.1
no_corr_cols = corr_with_target[corr_with_target < threshold].index.tolist()


print(f" Low / No Correlation Columns  (Total {len(no_corr_cols)}):")
print(no_corr_cols)


no_corr_df = pd.DataFrame({
    'Feature': no_corr_cols,
    'Correlation_Value': df.corr()['status'][no_corr_cols].values
}).sort_values(by='Correlation_Value', key=abs)

print("\n--- Low Correlation Values ---")
print(no_corr_df.to_string(index=False))

**HYPOTHYSIS TEST**

In [ ]:
import pandas as pd
from scipy import stats


hyp_results = []


placed_mask = df['status'] == 1  # Encoded status = 1 (Placed)

for col in no_corr_cols:
    unique_vals = df[col].nunique()

    # Categorical Feature (Chi-Square Test)
    if unique_vals <= 5: # e.g: gender, company_tier, specialization
        contingency_table = pd.crosstab(df[col], df['status'])
        stat, p_val, _, _ = stats.chi2_contingency(contingency_table)
        test_used = "Chi-Square"
    # Continuous / Numerical Feature (Two-Sample t-Test)
    else:
        group1 = df[placed_mask][col]
        group2 = df[~placed_mask][col]
        stat, p_val = stats.ttest_ind(group1, group2, nan_policy='omit')
        test_used = "t-Test"

    # P-value >= 0.05  (Not Significant)
    decision = "Reject (Drop Column)" if p_val >= 0.05 else "Keep Column"

    hyp_results.append({
        'Feature': col,
        'Test_Type': test_used,
        'P-Value': round(p_val, 5),
        'Decision': decision
    })

# Results DataFrame
hyp_df = pd.DataFrame(hyp_results)
print("=== Low Correlation Columns - Hypothesis Test Results ===")
print(hyp_df.to_string(index=False))

In [ ]:
# Reject column remove
cols_to_drop = hyp_df[hyp_df['Decision'].str.contains('Reject')]['Feature'].tolist()
df_final = df.drop(columns=cols_to_drop)

In [ ]:
df_final.shape

In [ ]:
# Features (X) and  Target (y)
X = df_final.drop(columns=['status'])
y = df_final['status']

In [ ]:
# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Feature Standardization (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1)
}

In [ ]:
model_results = []

for name, model in models.items():
    # Model Training
    model.fit(X_train_scaled, y_train)

    # Prediction
    y_pred = model.predict(X_test_scaled)

    # Metrics calculation
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    model_results.append({
        'Model': name,
        'Accuracy': round(acc * 100, 2),
        'Precision': round(prec * 100, 2),
        'Recall': round(rec * 100, 2),
        'F1-Score': round(f1 * 100, 2)
    })

In [ ]:
results_df = pd.DataFrame(model_results).sort_values(by='Accuracy', ascending=False)
print("===  ML Model Comparison Results ===")
print(results_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

best_model = XGBClassifier(random_state=42, eval_metric='logloss')
best_model.fit(X_train_scaled, y_train)

# 2. Feature Importance table creat
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# 3.  (Top 10 Features Graph)
plt.figure(figsize=(10, 5))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(10), palette='mako')
plt.title('Key Drivers of Placement Success (Top 10 Features)')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()